# 08 — Movie Recommendation Testing and Artifact Export

**Goal:** run end-to-end checks, expose clean functions for the app, and save a compact set of model artifacts. Run Notebooks 01–06 first; Notebook 07 is recommended for evaluation.

## 1. Load data and model artifacts

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Keep paths consistent whether the notebook is launched from the project root or 02_Notebooks.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "02_Notebooks" else CWD
DATA_RAW = PROJECT_ROOT / "01_Data" / "Raw"
DATA_PROCESSED = PROJECT_ROOT / "01_Data" / "Processed"
MODELS = PROJECT_ROOT / "03_Models"
VISUALS = PROJECT_ROOT / "05_Visualizations"
for folder in [DATA_RAW, DATA_PROCESSED, MODELS, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

import joblib
movies = pd.read_pickle(DATA_PROCESSED / "movies_processed.pkl").reset_index(drop=True)
ratings = pd.read_csv(DATA_PROCESSED / "ratings_processed.csv")
content = joblib.load(MODELS / "content_artifacts.joblib")
collab = joblib.load(MODELS / "collaborative_artifacts.joblib")
tfidf = content["tfidf_vectorizer"]
content_matrix = content["content_matrix"]
content_nn = content["content_nn"]
user_item = collab["user_item"]
item_similarity = collab["item_similarity"]
user_to_idx = collab["user_to_idx"]
rated_movie_ids = collab["rated_movie_ids"]

Project root: D:\DataScience\notebooks\CineMatch_Ready_Notebooks


## 2. Define app-ready functions

These functions return Pandas DataFrames, making them straightforward to display in Streamlit.

In [2]:
def search_movie_catalogue(query, limit=20):
    query=str(query).strip().lower()
    if not query:
        return movies.iloc[0:0][["movieId","title","genres","release_year","average_rating"]]
    mask=movies.title.str.lower().str.contains(query,regex=False,na=False)
    return movies.loc[mask,["movieId","title","genres","release_year","average_rating"]].head(limit)

def similar_movies_by_id(movie_id, top_n=10):
    matches=movies.index[movies.movieId==int(movie_id)]
    if len(matches)==0:
        return pd.DataFrame(columns=["movieId","title","genres","similarity"])
    idx=int(matches[0])
    distances,indices=content_nn.kneighbors(content_matrix[idx],
        n_neighbors=min(top_n+1,len(movies)))
    out=[]
    for d,j in zip(distances[0],indices[0]):
        row=movies.iloc[int(j)]
        if int(row.movieId)==int(movie_id): continue
        out.append({"movieId":int(row.movieId),"title":row.title,
                    "genres":row.genres,"similarity":float(1-d),
                    "average_rating":float(row.average_rating)})
        if len(out)>=top_n: break
    return pd.DataFrame(out)

def collaborative_recommendations(user_id, top_n=10):
    if int(user_id) not in user_to_idx:
        return pd.DataFrame(columns=["movieId","title","predicted_score"])
    ui=user_to_idx[int(user_id)]
    a,b=user_item.indptr[ui],user_item.indptr[ui+1]
    seen_cols=user_item.indices[a:b]; seen_ratings=user_item.data[a:b]
    scores=np.zeros(len(rated_movie_ids)); weights=np.zeros(len(rated_movie_ids))
    for col,rating in zip(seen_cols,seen_ratings):
        sims=item_similarity[:,col].toarray().ravel()
        scores+=sims*rating; weights+=np.abs(sims)
    pred=np.divide(scores,weights,out=np.zeros_like(scores),where=weights>0)
    pred[seen_cols]=-np.inf
    order=np.argsort(pred)[::-1]
    out=[]
    for col in order:
        if not np.isfinite(pred[col]) or weights[col]==0: continue
        out.append({"movieId":int(rated_movie_ids[col]),"predicted_score":float(pred[col])})
        if len(out)>=top_n: break
    return pd.DataFrame(out).merge(movies[["movieId","title","genres","average_rating"]],
                                   on="movieId",how="left")

## 3. End-to-end test cases

The tests below check known titles, unknown titles, known users, unknown users, empty input, and the rule that a user's previously rated movies are excluded.

In [3]:
# Search tests
assert search_movie_catalogue("").empty
assert not search_movie_catalogue("Toy Story").empty
assert search_movie_catalogue("a title that cannot exist 987654").empty

# Content tests
toy_matches=search_movie_catalogue("Toy Story")
if not toy_matches.empty:
    content_test=similar_movies_by_id(int(toy_matches.iloc[0].movieId),10)
    assert len(content_test)<=10
    assert int(toy_matches.iloc[0].movieId) not in set(content_test.movieId)
    display(content_test)
else:
    print("Toy Story not found in this version of the dataset.")

# User tests
known_uid=int(ratings.userId.iloc[0])
collab_test=collaborative_recommendations(known_uid,10)
seen=set(ratings.loc[ratings.userId==known_uid,"movieId"].astype(int))
assert not (set(collab_test.movieId.astype(int)) & seen) if not collab_test.empty else True
display(collab_test)
unknown_user_result=collaborative_recommendations(-123456,10)
assert unknown_user_result.empty
print("All basic tests passed. Example known user:",known_uid)

,movieId,title,genres,similarity,average_rating
0,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.424833,4.109091
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.392092,3.860825
2,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.283921,3.516304
3,13,Balto (1995),Adventure|Animation|Children,0.261937,3.125000
4,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,0.231144,2.500000
5,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.224983,3.871212
6,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.221189,3.023810
7,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,0.220016,2.500000
8,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,0.216590,3.244444
9,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,0.214876,3.450000


,movieId,predicted_score,title,genres,average_rating
0,193587,5.0,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,3.5
1,165635,5.0,The Thinning (2016),Thriller,1.0
2,175475,5.0,The Emoji Movie (2017),Animation|Children|Comedy,0.5
3,175585,5.0,Shot Caller (2017),Action|Crime|Drama|Thriller,3.5
4,44633,5.0,"Devil and Daniel Johnston, The (2005)",Documentary,4.5
5,2896,5.0,Alvarez Kelly (1966),Western,3.0
6,103210,5.0,Fullmetal Alchemist: The Sacred Star of Milos ...,Action|Adventure|Animation,2.0
7,1140,5.0,Entertaining Angels: The Dorothy Day Story (1996),Drama,5.0
8,193585,5.0,Flint (2017),Drama,3.5
9,177285,5.0,Sword Art Online The Movie: Ordinal Scale (2017),Action|Adventure|Animation|Fantasy|Sci-Fi,3.0


All basic tests passed. Example known user: 1


## 4. Save final app artifacts

The app needs the processed catalogue, content vectorizer/matrix/neighbour model, and collaborative sparse matrices/mappings. Joblib preserves Python objects and sparse matrices.

In [4]:
final_artifacts={
    "movies":movies,
    "tfidf_vectorizer":tfidf,
    "content_matrix":content_matrix,
    "content_nn":content_nn,
    "user_item":user_item,
    "item_similarity":item_similarity,
    "user_ids":collab["user_ids"],
    "rated_movie_ids":rated_movie_ids,
    "user_to_idx":user_to_idx,
    "movie_to_idx":collab["movie_to_idx"],
    "hybrid_config":{"content_weight":0.4,"collaborative_weight":0.6}
}
artifact_path=MODELS/"cinematch_artifacts.joblib"
joblib.dump(final_artifacts,artifact_path,compress=3)
print("Saved final artifact:",artifact_path)
print("Artifact size (MB):",round(artifact_path.stat().st_size/(1024**2),2))

Saved final artifact: D:\DataScience\notebooks\CineMatch_Ready_Notebooks\03_Models\cinematch_artifacts.joblib
Artifact size (MB): 208.78


## 5. Generate a project summary for the app/README

In [5]:
project_summary={
    "project":"CineMatch – Intelligent Movie Recommendation System",
    "movies":int(movies.movieId.nunique()),
    "users":int(ratings.userId.nunique()),
    "ratings":int(len(ratings)),
    "genres":int(movies.genre_list.explode().nunique()),
    "algorithms":["Content-Based Filtering","Item-Item Collaborative Filtering","Weighted Hybrid (where enabled)"],
    "dataset":"MovieLens Latest Small (GroupLens)"
}
summary_path=PROJECT_ROOT/"project_summary.json"
import json
summary_path.write_text(json.dumps(project_summary,indent=2),encoding="utf-8")
print(json.dumps(project_summary,indent=2))

{
  "project": "CineMatch \u2013 Intelligent Movie Recommendation System",
  "movies": 9742,
  "users": 610,
  "ratings": 100836,
  "genres": 19,
  "algorithms": [
    "Content-Based Filtering",
    "Item-Item Collaborative Filtering",
    "Weighted Hybrid (where enabled)"
  ],
  "dataset": "MovieLens Latest Small (GroupLens)"
}


## Completion checklist

- [ ] Notebook 01 downloaded and inspected the source data.
- [ ] Notebook 02 created processed movie and rating tables.
- [ ] Notebook 03 produced EDA outputs.
- [ ] Notebook 04 saved content recommendation artifacts.
- [ ] Notebook 05 saved collaborative artifacts.
- [ ] Notebook 06 saved hybrid configuration.
- [ ] Notebook 07 evaluated ranking quality without random interaction leakage.
- [ ] Notebook 08 passed tests and saved `03_Models/cinematch_artifacts.joblib`.

**Next development stage:** build `04_App/app.py` with Streamlit and load `cinematch_artifacts.joblib`. The notebooks are the model-development pipeline; the Streamlit app is a separate deliverable.